In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
TOKENIZER_NAME = "distilbert-base-uncased"
MODEL_NAME = "distilbert-base-uncased"           # tokenizer-file error on some bert-tiny mirrors.
                                          # Same vocabulary, so nothing breaks.      # the actual model used for embeddings
MAX_LEN = 128
BATCH_SIZE = 32
SAMPLE_SIZE = None

In [3]:
print("MODEL_NAME is currently:", MODEL_NAME)

MODEL_NAME is currently: distilbert-base-uncased


In [4]:
df = pd.read_csv("/content/cleaned_data.csv")
print(f"Loaded {len(df)} rows")

if SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f"Using a sample of {len(df)} rows")

X_text = df["review"].astype(str)
y = df["label"].astype(int)

trans_X_train_text, trans_X_test_text, trans_y_train, trans_y_test = train_test_split(
    X_text, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train: {len(trans_X_train_text)}  Test: {len(trans_X_test_text)}")

Loaded 49582 rows
Train: 37186  Test: 12396


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.eval()
bert_model.to(device)
print(f"Loaded {MODEL_NAME} (tokenizer from {TOKENIZER_NAME}) on {device}")


Running on: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded distilbert-base-uncased (tokenizer from distilbert-base-uncased) on cuda


In [6]:
def get_embeddings(texts):
    """One forward pass per review, no gradients -- fast and memory-light."""
    texts = texts.reset_index(drop=True)
    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding"):
            batch = texts[i:i + BATCH_SIZE].tolist()
            encoded = tokenizer(
                batch, padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors="pt"
            ).to(device)
            output = bert_model(**encoded)
            cls_embeddings = output.last_hidden_state[:, 0, :]  # [CLS] token summary
            embeddings.append(cls_embeddings.cpu().numpy())
    return np.vstack(embeddings)

In [7]:
print("\nExtracting training embeddings...")
trans_X_train_embed = get_embeddings(trans_X_train_text)

print("\nExtracting test embeddings...")
trans_X_test_embed = get_embeddings(trans_X_test_text)


Extracting training embeddings...


Embedding:   0%|          | 0/1163 [00:00<?, ?it/s]


Extracting test embeddings...


Embedding:   0%|          | 0/388 [00:00<?, ?it/s]

In [8]:
classifier = Sequential([
    Input(shape=(trans_X_train_embed.shape[1],)),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid"),
])

In [9]:
classifier.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
classifier.summary()

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

classifier.fit(
    trans_X_train_embed, trans_y_train,
    validation_split=0.2,
    epochs=20,          # EarlyStopping cuts this short once it stops improving
    batch_size=32,
    callbacks=[early_stop],
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 106,753 (417.00 KB)

 Trainable params: 106,753 (417.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
930/930 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.7765 - loss: 0.4701 - val_accuracy: 0.8026 - val_loss: 0.4242
Epoch 2/20
930/930 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8023 - loss: 0.4294 - val_accuracy: 0.8106 - val_loss: 0.4112
Epoch 3/20
930/930 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8063 - loss: 0.4201 - val_accuracy: 0.8072 - val_loss: 0.4116
Epoch 4/20
930/930 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8081 - loss: 0.4143 - val_accuracy: 0.8119 - val_loss: 0.4099
Epoch 5/20
930/930 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8120 - loss: 0.4093 - val_accuracy: 0.8115 - val_loss: 0.4065
Epoch 6/20
930/930 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8141 - loss: 0.4069 - val_accuracy: 0.8120 - val_loss: 0.4064
Epoch 7/20
930/930 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8150 - loss: 0.4038 - val_accuracy: 0.8146 - val_loss: 0.4005
Epoch 8/20
930/930 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8158 - loss: 0.4008 - val_accuracy: 0.

In [10]:
trans_y_pred_prob = classifier.predict(trans_X_test_embed)
trans_y_pred = (trans_y_pred_prob > 0.5).astype(int).ravel()

print("\nAccuracy: ", accuracy_score(trans_y_test, trans_y_pred))
print("Precision:", precision_score(trans_y_test, trans_y_pred))
print("Recall:   ", recall_score(trans_y_test, trans_y_pred))
print("F1 score: ", f1_score(trans_y_test, trans_y_pred))
print()
print(classification_report(trans_y_test, trans_y_pred, target_names=["Negative", "Positive"]))

388/388 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

Accuracy:  0.8184091642465311
Precision: 0.8155802861685215
Recall:    0.8246262658736537
F1 score:  0.8200783310686596

              precision    recall  f1-score   support

    Negative       0.82      0.81      0.82      6175
    Positive       0.82      0.82      0.82      6221

    accuracy                           0.82     12396
   macro avg       0.82      0.82      0.82     12396
weighted avg       0.82      0.82      0.82     12396



In [11]:
classifier.save("transformer_classifier.keras")
tokenizer.save_pretrained("bert_tokenizer")
bert_model.save_pretrained("bert_base")

print("\nSaved transformer_classifier.keras, bert_tokenizer/, bert_base/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved transformer_classifier.keras, bert_tokenizer/, bert_base/


In [12]:
!zip -rq transformer_artifacts.zip transformer_classifier.keras bert_tokenizer bert_base
from google.colab import files
files.download("transformer_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>